In [1]:
%load_ext autoreload
%autoreload 2

## Usage examples

In [23]:
import os
import sys
import sqlite3
import pandas as pd
from pathlib import Path

ROOT = Path(os.getcwd()).parent.parent.parent
sys.path.append(ROOT / "common_code")

In [3]:

# verb transactions database
TRANSACTION_DB = ROOT / "databases/v33_data.db"
#TRANSACTION_DB = Path("./example_data/transactions.db")

# result database with ner and timex data
NER_TIMEX_DB= "./example_data/ner_timex.db"



In [13]:
con = sqlite3.connect(TRANSACTION_DB)
con.row_factory = sqlite3.Row 
cur = con.cursor()
cur.execute(f'ATTACH DATABASE "{NER_TIMEX_DB}" AS ner_timex')


### Fetching TIMEX of verb transaction heads


In [30]:
sql = """
SELECT th.id as head_id, th.sentence_id, th.loc, th.form, th.deprel, timex.*, th.phrase
FROM ner_timex.timex AS timex
INNER JOIN transaction_head AS th ON timex.sentence_id = th.sentence_id AND timex.loc = th.loc
--WHERE th.deprel NOT IN ("acl", "root")
LIMIT 100
"""
res = [dict(r) for r in cur.execute(sql).fetchall()]
df = pd.DataFrame.from_dict(res)
df.head()


,head_id,sentence_id,loc,form,deprel,id,timex_id,timex_type,part_of_interval,timex_members,phrase
0,769,428,1,Läinud,acl,42,t1,DATE,,2,Läinud
1,1004,564,23,möödunud,acl,60,t2,DATE,,2,möödunud
2,2550,1409,1,Läinud,acl,180,t1,DATE,,2,Läinud
3,4468,2684,12,läinud,acl,293,t1,DATE,,2,erilehes veel läinud
4,4537,2725,1,Möödunud,acl,300,t1,DATE,,2,Möödunud


### Fetching TIMEX of verb transaction rows

In [31]:
sql = """
SELECT tr.id as row_id, th.sentence_id, tr.loc, tr.form, tr.deprel, timex.*, th.phrase
FROM ner_timex.timex AS timex
INNER JOIN transaction_head as th ON timex.sentence_id = th.sentence_id
INNER JOIN transaction_row AS tr ON tr.head_id = th.id AND timex.loc = tr.loc
LIMIT 10
"""
res = [dict(r) for r in cur.execute(sql).fetchall()]
df = pd.DataFrame.from_dict(res)
df.head()

,row_id,sentence_id,loc,form,deprel,id,timex_id,timex_type,part_of_interval,timex_members,phrase
0,1,3,3,lõpus,obl,1,t1,DATE,,3,lõpus toimus Türi 1.
1,37,20,6,nüüd,advmod,2,t1,DATE,,1,nüüd täidavad palad tantsupõrandaid
2,52,24,5,aastate,obl,3,t1,DATE,,3,Ma jõudsin tantsumuusikani aastate algul poolj...
3,53,24,6,algul,case,4,t1,DATE,,3,Ma jõudsin tantsumuusikani aastate algul poolj...
4,129,54,3,ajal,obl,5,t1,DATE,,2,Olen ajal juhutöödest loobunud


In [7]:
#cur.close()